# 05 — Model 1: Logistic Regression


> **Notebook 5 of 11** — part of the *Heart Disease Detection using Explainable AI* project.
> Run the notebooks **in order**, from 01 to 11.

---

## 🎯 How Logistic Regression works, explained simply

Imagine plotting every patient on a graph. Logistic Regression draws **one straight boundary** and
says: *everyone on this side is probably healthy, everyone on that side probably has heart
disease.* It then converts the distance from that line into a probability between 0 and 1.

| Strength | Weakness |
|---|---|
| Simple, fast, completely transparent | Cannot capture curved relationships |
| A doctor can read its coefficients directly | Assumes each feature acts independently |

It is the **honest baseline** every project needs: if a complicated model cannot beat a straight
line, the complication was not worth it.

## ⚙️ The setting we tune: `C`

`C` controls how strict the model is.

* **Small `C`** (0.01) = strict. Keeps the line simple, refuses to over-react to noise.
* **Large `C`** (10) = relaxed. Bends to fit training data closely, risking **overfitting** —
  memorising instead of learning.

`GridSearchCV` will try four values and keep whichever genuinely scores best.

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve, classification_report)

warnings.filterwarnings("ignore"); sns.set_style("whitegrid")
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
DATA, MODELS = "../data", "../models"

# --- load what notebook 04 prepared ---
prep = np.load(f"{DATA}/prepared.npz", allow_pickle=True)
FEATURES = list(prep["features"])
X_train, X_test = prep["X_train"], prep["X_test"]
y_train, y_test = prep["y_train"], prep["y_test"]
scaler = joblib.load(f"{MODELS}/scaler.pkl")

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)
cv_plan = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"Training patients : {len(X_train):,}")
print(f"Test patients     : {len(X_test):,}")
print(f"Features          : {FEATURES}")

from sklearn.linear_model import LogisticRegression

In [ ]:
def save_results(model_name, model, best_params, cv_scores, seconds):
    """Grade the model on the SEALED test set and append the scores to models/metrics.json."""
    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    keep = max(1, len(fpr) // 200)

    entry = {
        "accuracy":  float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred)),
        "recall":    float(recall_score(y_test, y_pred)),
        "f1":        float(f1_score(y_test, y_pred)),
        "roc_auc":   float(roc_auc_score(y_test, y_proba)),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "fpr": fpr[::keep].tolist(), "tpr": tpr[::keep].tolist(),
        "cv_scores": [float(s) for s in cv_scores],
        "cv_mean": float(np.mean(cv_scores)), "cv_std": float(np.std(cv_scores)),
        "best_params": {k: str(v) for k, v in best_params.items()},
        "train_seconds": round(seconds, 1),
    }

    path = f"{MODELS}/metrics.json"
    all_metrics = json.load(open(path)) if os.path.exists(path) else {}
    all_metrics[model_name] = entry
    json.dump(all_metrics, open(path, "w"), indent=2)

    print(f"\n{'=' * 58}\n  {model_name}\n{'=' * 58}")
    print(f"  Accuracy   : {entry['accuracy']:.4f}   (out of 100 patients, "
          f"{entry['accuracy']*100:.0f} labelled correctly)")
    print(f"  Precision  : {entry['precision']:.4f}   (when we say 'disease', we are right this often)")
    print(f"  Recall     : {entry['recall']:.4f}   (of all truly sick, we caught this many)")
    print(f"  F1 Score   : {entry['f1']:.4f}   (balance of precision and recall)")
    print(f"  ROC-AUC    : {entry['roc_auc']:.4f}   (0.5 = coin toss, 1.0 = perfect)")
    print(f"  CV accuracy: {entry['cv_mean']:.4f} +/- {entry['cv_std']:.4f}")
    return y_proba

print("Helper ready.")

## 1. Hyperparameter tuning with GridSearchCV

In [ ]:
print("Tuning Logistic Regression... (about 20-40 seconds)")
start = time.time()

grid = GridSearchCV(
    estimator=LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    param_grid={"C": [0.01, 0.1, 1.0, 10.0]},
    cv=cv_plan, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print(f"\nBest setting : {grid.best_params_}")
print(f"Best CV ROC-AUC : {grid.best_score_:.4f}")
print(f"Took {time.time()-start:.0f} seconds")

pd.DataFrame({"C value": grid.cv_results_["param_C"],
              "ROC-AUC": grid.cv_results_["mean_test_score"].round(4)}
            ).sort_values("ROC-AUC", ascending=False)

Notice how little difference the `C` value makes here. That is itself informative: with only 11
features and 54,000 patients, there is not much room to overfit, so regularisation strength barely
matters. A small `C` wins by a hair, so we take the simpler model.

## 2. Cross-validation — is the score stable?

We split the training data into 5 parts, train 5 times, and check the score every time. If all
five numbers are close together, the result is real. If they jump around, we got lucky once.

In [ ]:
base_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, **grid.best_params_)
cv_scores = cross_val_score(base_model, X_train_scaled, y_train,
                            cv=cv_plan, scoring="accuracy", n_jobs=-1)

print("Accuracy on each of the 5 folds:", np.round(cv_scores, 4))
print(f"Average   : {cv_scores.mean():.4f}")
print(f"Variation : +/- {cv_scores.std():.4f}")
print("\nThe five numbers are nearly identical, so the result is stable and repeatable.")

## Why `CalibratedClassifierCV`?

A model's raw output is a **score**, not necessarily an honest probability. A model might output
0.80 for a group of patients of whom only 65% are really sick — the *ranking* is right but the
*number* is exaggerated, and different algorithms exaggerate differently.

That matters for us, because the website shows all three percentages **side by side**. If one says
45% and another says 78%, the user cannot tell whom to believe.

**Isotonic calibration** learns a correction curve on held-out folds and re-maps every score so
that among patients given 70%, roughly 70% really are sick. Notebook 08 measures how well this
worked.

In [ ]:
final_model = CalibratedClassifierCV(base_model, cv=5, method="isotonic")
final_model.fit(X_train_scaled, y_train)

proba = save_results("Logistic Regression", final_model,
                     grid.best_params_, cv_scores, time.time() - start)

## 3. The detailed classification report

In [ ]:
print(classification_report(y_test, final_model.predict(X_test_scaled),
                            target_names=["Healthy", "Heart disease"], digits=4))

In [ ]:
cm = confusion_matrix(y_test, final_model.predict(X_test_scaled))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Predicted\nHealthy", "Predicted\nDisease"],
            yticklabels=["Actually\nHealthy", "Actually\nDisease"])
ax.set_title("Logistic Regression — confusion matrix")
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Correctly found sick      : {tp:,}")
print(f"Correctly cleared healthy : {tn:,}")
print(f"False alarms              : {fp:,}")
print(f"MISSED sick patients      : {fn:,}   <-- the dangerous mistake")

## 4. 🎁 The bonus: we can read its reasoning directly

This is Logistic Regression's superpower. Each feature gets one **coefficient** — a single number
saying how much it pushes the answer towards heart disease.

* **Positive** coefficient → pushes towards disease
* **Negative** coefficient → pushes towards healthy

Because we scaled all features to the same range in notebook 04, these coefficients are directly
comparable to each other.

In [ ]:
plain = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE,
                           **grid.best_params_).fit(X_train_scaled, y_train)

coef = pd.DataFrame({"Feature": FEATURES,
                     "Coefficient": plain.coef_[0]}).sort_values("Coefficient")

fig, ax = plt.subplots(figsize=(9, 5))
colours = ["#3B82F6" if v < 0 else "#EF4444" for v in coef.Coefficient]
ax.barh(coef.Feature, coef.Coefficient, color=colours)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Logistic Regression coefficients\nred = towards heart disease,"
             " blue = towards healthy")
ax.set_xlabel("Coefficient")
plt.tight_layout(); plt.show()

print(coef.sort_values("Coefficient", ascending=False).to_string(index=False))

**Read that chart against notebook 03.** Blood pressure, age and cholesterol have the largest
positive coefficients; being physically active has a negative one. That is exactly what our EDA
found by simple counting, and exactly what a doctor would tell you. Two independent methods
agreeing is a good sign.

In [ ]:
joblib.dump(final_model, f"{MODELS}/logistic_regression.pkl")
np.save(f"{MODELS}/proba__logistic_regression.npy", proba)
print(f"Saved {MODELS}/logistic_regression.pkl")

---
## ✅ Summary

* Tuned `C` with GridSearchCV, validated with 5-fold cross-validation.
* Calibrated the probabilities with isotonic regression.
* Accuracy around **73%**, ROC-AUC around **0.79**.
* Its coefficients independently confirm what notebook 03's EDA found.

### ▶️ Next: `06_Random_Forest.ipynb`